# 日报数据与统计口径验收

## TL;DR

技术数据验收通过；正式统计仍需复核 106 条分类、323.5 小时。业务总体覆盖未评估，因为当前库是部分测试语料，不能从最早/最晚上传日期推断完整月份。

## Context & Methods

本笔记读取 `scripts/audit_data_acceptance.py` 生成的只读验收快照，并复核数据库约束、事实视图对账、operation 统计就绪率、逐日报问题分布，以及 2026 年 6 月外部工作量表与当前上传范围的可比性。正式统计采用 `vw_rig_production_timeline.statistical_hours`。

In [1]:
from pathlib import Path
from collections import Counter
import csv
import json

snapshot = json.loads(Path('acceptance_snapshot.json').read_text(encoding='utf-8'))
summary = snapshot['summary']
summary

{'generated_on': '2026-07-18',
 'technical_acceptance_status': 'PASS',
 'official_statistics_status': 'REVIEW_REQUIRED',
 'business_universe_coverage_status': 'NOT_ASSESSED',
 'report_count': 172,
 'operation_count': 1404,
 'parsed_hours': 3967.5,
 'official_hours': 3644.0,
 'pending_classification_hours': 323.5,
 'statistics_readiness_percent': 91.8,
 'clock_derived_operation_rows': 1,
 'report_acceptance_counts': {'PASS': 133, 'REVIEW': 39},
 'source_status_counts': {'RESOLVED': 171, 'IDENTICAL_DUPLICATES': 1}}

## Data

快照覆盖 172 份日报、1,404 条 operation，并把解析总工时、正式统计工时和待复核工时分开。

In [2]:
snapshot['by_report_type']

[{'report_type': 'completion',
  'report_count': 52,
  'operation_count': 392,
  'parsed_hours': 1218.0,
  'official_hours': 1112.0,
  'pending_hours': 106.0},
 {'report_type': 'drilling',
  'report_count': 79,
  'operation_count': 686,
  'parsed_hours': 1830.5,
  'official_hours': 1694.0,
  'pending_hours': 136.5},
 {'report_type': 'move',
  'report_count': 1,
  'operation_count': 3,
  'parsed_hours': 24.0,
  'official_hours': 24.0,
  'pending_hours': 0.0},
 {'report_type': 'workover',
  'report_count': 40,
  'operation_count': 323,
  'parsed_hours': 895.0,
  'official_hours': 814.0,
  'pending_hours': 81.0}]

## Results

首先验证事实表、分类表和视图行数完全一致，且没有孤儿事实、重复 operation 键、开放错误级问题或开放有效期重叠。

In [3]:
[(row['check_code'], row['status']) for row in snapshot['checks']]

[('REPORT_FACT_1_TO_1', 'PASS'),
 ('REPORT_SUMMARY_1_TO_1', 'PASS'),
 ('OPERATION_CLASSIFICATION_1_TO_1', 'PASS'),
 ('STRUCTURED_VIEW_ROW_RECONCILIATION', 'PASS'),
 ('TIMELINE_VIEW_ROW_RECONCILIATION', 'PASS'),
 ('NO_ORPHAN_OR_DUPLICATE_FACTS', 'PASS'),
 ('NO_OPEN_ERROR_ISSUES', 'PASS'),
 ('NO_OPEN_RELATIONSHIP_OVERLAPS', 'PASS'),
 ('COMPLETED_TRANSLATIONS_HAVE_CONTENT', 'PASS'),
 ('SOURCE_PDF_PROVENANCE', 'PASS')]

In [4]:
status_counts = Counter(row['acceptance_status'] for row in snapshot['reports'])
issue_counts = Counter(code for row in snapshot['reports'] for code in row['issue_codes'])
{'report_statuses': dict(status_counts), 'issue_codes': dict(issue_counts)}

{'report_statuses': {'PASS': 133, 'REVIEW': 39},
 'issue_codes': {'BOUNDARY_HOURS_NOT_24': 14, 'CLASSIFICATION_PENDING': 25}}

In [5]:
with Path('external_monthly_workload_reconciliation.csv').open(encoding='utf-8-sig') as handle:
    external_rows = list(csv.DictReader(handle))
external_summary = {
    'external_rigs': len(external_rows),
    'database_rigs_with_june_data': sum(int(row['database_report_count']) > 0 for row in external_rows),
    'external_total_hours': round(sum(float(row['external_total_hours']) for row in external_rows), 1),
    'database_parsed_hours_in_external_rigs': round(sum(float(row['database_parsed_hours']) for row in external_rows), 1),
    'total_aligned_rigs': sum(row['reconciliation_status'] == 'TOTAL_HOURS_ALIGNED' for row in external_rows),
}
external_summary

{'external_rigs': 15,
 'database_rigs_with_june_data': 9,
 'external_total_hours': 10183.0,
 'database_parsed_hours_in_external_rigs': 1529.0,
 'total_aligned_rigs': 0}

## Takeaways

- 技术层可验收：来源、结构、时效、翻译、主数据关系及视图对账均通过。
- 统计层尚未最终验收：323.5 小时分类待确认，接口必须持续单列，不得静默进入 P/SC/NPT 或效率。
- 当前 6 月库内范围与整月工作量表不相等，因此不做总量相等判定；待上传范围一致后再执行正式月度对账。
- 下一业务动作是按工时和影响排序完成分类确认，随后重跑同一验收脚本。